In [ ]:
"""
IEEE BigData - ADAPTIVE MULTI-CRITERIA FEDERATED RIDGE & CONGESTION-AWARE ROUTING
  * Per-target alpha selection: Ridge eigenvalue decomposition optimizes regularisation independently per telemetry channel.
  * Adaptive Conformal Inference (ACI): Rolling quantile update prevents coverage breakdown under non-stationary drift.
  * Rescaled Ping Log-Predictor: Lowered gate hurdle (2.0%) tailored for heavy-tailed log-ping telemetry.
  * Multi-agent space-time conflict resolution: Dynamic graph reservation avoids AP congestion and agent collisions.
  * Non-overlapping temporal rolling-origins (0.40, 0.55, 0.70, 0.85) ensuring uncompromised Diebold-Mariano tests.
"""

from __future__ import annotations

import csv
import heapq
import itertools
import json
import logging
import math
import os
import random
import re
import sys
import time
from dataclasses import dataclass
from pathlib import Path
from typing import Any, Dict, List, Optional, Sequence, Tuple

import numpy as np

if not hasattr(np, "matrix"):
    from numpy.matrixlib.defmatrix import matrix as _numpy_matrix_type
    np.matrix = _numpy_matrix_type

import pandas as pd
from scipy.spatial import cKDTree, Delaunay, QhullError
from scipy.sparse.csgraph import connected_components
from scipy.sparse import lil_matrix
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
from scipy.stats import t as _student_t

# ============================================================================
# 0. CONFIGURATION & LOGGING
# ============================================================================

SEEDS = [int(s) for s in os.environ.get("AGV_SEEDS", os.environ.get("AGV_SEED", "42")).split(",") if s.strip()]
SEED = SEEDS[0]
HORIZON = max(1, int(os.environ.get("AGV_HORIZON", "1")))
BOOTSTRAP_REPLICATES = int(os.environ.get("AGV_BOOTSTRAP_REPLICATES", "300"))
BOOTSTRAP_BLOCK_LENGTH = int(os.environ.get("AGV_BOOTSTRAP_BLOCK_LENGTH", "5"))

def _find_project_root() -> Path:
    env_root = os.environ.get("AGV_PROJECT_ROOT", "").strip()
    candidates = []
    if env_root:
        candidates.append(Path(env_root).expanduser().resolve())
    try:
        candidates.extend([Path(__file__).resolve().parent, *Path(__file__).resolve().parent.parents])
    except NameError:
        pass
    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    seen = set()
    for root in candidates:
        root = root.resolve()
        if root in seen:
            continue
        seen.add(root)
        if (root / "data" / "FL_WI_FI_CLIENTS" / "varlist.txt").exists():
            return root
    return candidates[0] if candidates else cwd

PROJECT_ROOT = _find_project_root()
BASE_FOLDER = PROJECT_ROOT / "data" / "FL_WI_FI_CLIENTS"
VARLIST_PATH = BASE_FOLDER / "varlist.txt"
CLIENT_FILES = [
    BASE_FOLDER / "out_client1.csv",
    BASE_FOLDER / "out_client2.csv",
    BASE_FOLDER / "out_client3.csv",
]
RESULTS_DIR = BASE_FOLDER / "results_residual_federated_battery_wifi_v24_0_1"
LOG_PATH = RESULTS_DIR / "experiment_v24_0.log"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

logger = logging.getLogger("residual_agv_v24_0")
logger.setLevel(logging.INFO)
logger.handlers.clear()
formatter = logging.Formatter("%(asctime)s | %(levelname)s | %(message)s")
fh = logging.FileHandler(LOG_PATH, encoding="utf-8")
fh.setFormatter(formatter)
sh = logging.StreamHandler(sys.stdout)
sh.setFormatter(formatter)
logger.addHandler(fh)
logger.addHandler(sh)

AGGREGATION_FREQ = "1min"
LOOKBACK = 12
WIFI_LOOKBACK = 20

BLEND_STEP = 0.05
BLEND_SHRINK = 0.75
MIN_GATE_GAIN_PCT = 0.5
PING_GATE_MIN_GAIN_PCT = 2.0  # fine-tuned hurdle for log-space ping stability

WIFI_AGG_STATS = ("last", "min", "max", "std")
MIN_SEQUENCE_SAMPLES = 25
MIN_CALIBRATION_SAMPLES = 20
BATTERY_CALIBRATION_FRACTION = float(os.environ.get("AGV_BATTERY_CALIBRATION_FRACTION", "0.75"))
PING_ACF_MAX_LAG = max(1, int(os.environ.get("AGV_PING_ACF_MAX_LAG", "10")))
NEAR_CONSTANT_MAX_UNIQUE = 5
NEAR_CONSTANT_DOMINANT_FRACTION = 0.95

#non-overlapping rolling origins
ORIGINS = tuple(float(x) for x in os.environ.get("AGV_ORIGINS", "0.40,0.55,0.70,0.85").split(",") if x.strip())
TEST_SPAN = 0.15
FAST_MAPS = tuple(x.strip() for x in os.environ.get("AGV_FAST_MAPS", "flat5,flat10,sum20").split(",") if x.strip())
FAST_ALPHAS = np.array([1.0, 3.0, 10.0, 30.0, 100.0, 300.0, 1e3, 3e3, 1e4, 1e5])
FAST_FOLDS = max(2, int(os.environ.get("AGV_FAST_FOLDS", "4")))
FAST_MODEL_NAME = "FedRidge-Adaptive"

WIFI_REGRESSION_FIELDS = [
    "load_long", "load_mid", "load_short", "ping",
    "signal_noise", "signal_power", "signal_quality",
]
WIFI_ROLL_WINDOWS = (10, 30, 60)
WIFI_SLOPE_LAGS = (5, 10, 30)

TARGET_BATTERY = "fh6000ens_energy_signalsbattery_value_64"
NAV_X = "fh6000nns_natural_navigation_signalsxcoordinate_181"
NAV_Y_ALIASES = [
    "fh6000nns_natural_navigation_signalsycoordinate_182",
    "fh6000nns_natural_navigation_signalssycoordinate_182",
    "fh6000nns_natural_navigation_signalalsycoordinate_182",
]
NAV_SPEED = "fh6000nns_natural_navigation_signalsspeed_179"
NAV_HEADING = "fh6000nns_natural_navigation_signalsheading_172"

ACTION_FEATURES = [
    "candidate_dx", "candidate_dy", "candidate_distance",
    "candidate_speed", "candidate_x", "candidate_y",
]
WIFI_ACTION_FEATURES = [
    "candidate_dx", "candidate_dy", "candidate_distance",
    "candidate_cos_theta", "candidate_sin_theta",
    "candidate_speed", "candidate_delta_speed",
    "candidate_x", "candidate_y",
]

ROUTER_IDS = ("router1", "router2", "router3")
BASE_ROUTER_SOURCE_COLUMNS = {
    "load_long": "fh7000mikrotik_load_long_1",
    "load_mid": "fh7000mikrotik_load_mid_2",
    "load_short": "fh7000mikrotik_load_short_3",
    "ping": "fh7000mikrotik_ping_4",
    "reserve": "fh7000mikrotik_reserve_5",
    "signal_noise": "fh7000mikrotik_signal_noise_6",
    "signal_power": "fh7000mikrotik_signal_power_7",
    "signal_quality": "fh7000mikrotik_signal_quality_8",
    "uptime": "fh7000mikrotik_uptime_9",
    "wifi_state_1": "fh7000mikrotik_wifi_state_1_10",
}
WIFI_AUX_FIELDS = ("uptime", "reserve", "wifi_state_1")
ROUTER_SOURCE_COLUMNS = {"router1__" + k: v for k, v in BASE_ROUTER_SOURCE_COLUMNS.items()}

WIFI_FIELDS: List[str] = []
ALL_ROUTER_FIELDS: List[str] = []

def set_global_determinism(seed: int = SEED) -> None:
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)

def clean_name(s: str) -> str:
    s = re.sub(r"\[([^\]]+)\]\s*-?\s*", r"\1_", str(s).strip())
    s = re.sub(r"[^\w\s]", "", s)
    s = re.sub(r"[\s_]+", "_", s)
    return s.strip("_").lower()

def resolve_column(columns: Sequence[str], preferred: str, aliases: Sequence[str] = ()) -> Optional[str]:
    cset = set(columns)
    if preferred in cset:
        return preferred
    for a in aliases:
        if a in cset:
            return a
    p = preferred.lower()
    for c in columns:
        if p in c.lower():
            return c
    return None

def safe_float_series(s: pd.Series) -> pd.Series:
    return pd.to_numeric(s, errors="coerce").astype(float)

def rmse(y_true, y_pred) -> float:
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def safe_r2(y_true, y_pred) -> float:
    try:
        if len(np.unique(np.asarray(y_true))) < 2:
            return float("nan")
        return float(r2_score(y_true, y_pred))
    except Exception:
        return float("nan")

# ============================================================================
# 1. STATISTICAL VALIDATION & BOOTSTRAP
# ============================================================================

def _paired_block_bootstrap_ci(y_true, baseline_pred, model_pred, metric="mae", reps=None, block_len=None):
    y = np.asarray(y_true, dtype=float).reshape(-1)
    b = np.asarray(baseline_pred, dtype=float).reshape(-1)
    m = np.asarray(model_pred, dtype=float).reshape(-1)
    n = len(y)
    reps = BOOTSTRAP_REPLICATES if reps is None else int(reps)
    block_len = BOOTSTRAP_BLOCK_LENGTH if block_len is None else max(1, int(block_len))
    if n < max(10, 2 * block_len) or reps < 20:
        return (float("nan"), float("nan"))
    rng = np.random.default_rng(SEED + n + (0 if metric == "mae" else 1))
    n_blocks = int(np.ceil(n / block_len))
    starts_max = max(1, n - block_len + 1)
    deltas = np.empty(reps, dtype=float)
    for r in range(reps):
        starts = rng.integers(0, starts_max, size=n_blocks)
        idx = np.concatenate([np.arange(st, st + block_len) for st in starts])[:n]
        if metric == "mae":
            deltas[r] = np.mean(np.abs(y[idx] - b[idx])) - np.mean(np.abs(y[idx] - m[idx]))
        else:
            deltas[r] = np.sqrt(np.mean((y[idx] - b[idx]) ** 2)) - np.sqrt(np.mean((y[idx] - m[idx]) ** 2))
    lo, hi = np.quantile(deltas, [0.025, 0.975])
    return float(lo), float(hi)

def _dm_pvalue(base_err, model_err, h: int = 1) -> float:
    d = np.abs(np.asarray(base_err, dtype=float).reshape(-1)) - np.abs(np.asarray(model_err, dtype=float).reshape(-1))
    n = len(d)
    if n < 8 or not np.all(np.isfinite(d)) or np.allclose(d, 0.0):
        return float("nan")
    h = max(1, int(h))
    dc = d - float(d.mean())
    max_lag = min(n - 1, max(h - 1, int(BOOTSTRAP_BLOCK_LENGTH) - 1))
    long_run_var = float(np.dot(dc, dc) / n)
    for lag in range(1, max_lag + 1):
        gamma = float(np.dot(dc[lag:], dc[:-lag]) / n)
        weight = 1.0 - lag / (max_lag + 1.0)
        long_run_var += 2.0 * weight * gamma
    if not np.isfinite(long_run_var) or long_run_var <= 0:
        return float("nan")
    corr = (n + 1 - 2 * h + h * (h - 1) / n) / n
    if corr <= 0:
        return float("nan")
    stat = float(d.mean()) / math.sqrt(long_run_var / n) * math.sqrt(corr)
    return float(2.0 * _student_t.sf(abs(stat), df=n - 1))

def holm_adjust(p) -> np.ndarray:
    p = np.asarray(p, dtype=float)
    out = np.full_like(p, np.nan)
    idx = np.where(np.isfinite(p))[0]
    if len(idx) == 0:
        return out
    order = idx[np.argsort(p[idx])]
    m = len(order); running = 0.0
    for rank, i in enumerate(order):
        running = max(running, min(1.0, (m - rank) * p[i]))
        out[i] = running
    return out

IMPROVED_STATUSES = ("improved_both_metrics", "improved_MAE_RMSE_not_worse", "improved_RMSE_MAE_not_worse")

def metric_pair_status(y_true, baseline_pred, model_pred, tol: Optional[float] = None) -> Dict[str, Any]:
    y_true = np.asarray(y_true, dtype=float).reshape(-1)
    baseline_pred = np.asarray(baseline_pred, dtype=float).reshape(-1)
    model_pred = np.asarray(model_pred, dtype=float).reshape(-1)
    base_mae = float(mean_absolute_error(y_true, baseline_pred))
    model_mae = float(mean_absolute_error(y_true, model_pred))
    base_rmse = rmse(y_true, baseline_pred)
    model_rmse = rmse(y_true, model_pred)
    scale = max(base_mae, base_rmse, float(np.std(y_true)), 1e-3)
    effective_tol = float(tol) if tol is not None else max(1e-6, 1e-5 * scale)
    mae_delta, rmse_delta = base_mae - model_mae, base_rmse - model_rmse
    mae_better, mae_worse = mae_delta > effective_tol, mae_delta < -effective_tol
    rmse_better, rmse_worse = rmse_delta > effective_tol, rmse_delta < -effective_tol

    if base_mae <= effective_tol and base_rmse <= effective_tol:
        status = "baseline_error_near_zero"
    elif mae_better and rmse_better:
        status = "improved_both_metrics"
    elif mae_better and not rmse_worse:
        status = "improved_MAE_RMSE_not_worse"
    elif rmse_better and not mae_worse:
        status = "improved_RMSE_MAE_not_worse"
    elif (mae_better and rmse_worse) or (mae_worse and rmse_better):
        status = "mixed_tradeoff"
    elif mae_worse or rmse_worse:
        status = "degraded"
    else:
        status = "baseline_retained"

    mae_ci = _paired_block_bootstrap_ci(y_true, baseline_pred, model_pred, "mae")
    rmse_ci = _paired_block_bootstrap_ci(y_true, baseline_pred, model_pred, "rmse")
    significant = bool(np.isfinite(mae_ci[0]) and mae_ci[0] > 0.0)
    raw_status = status
    if status in IMPROVED_STATUSES and not significant:
        status = "improved_not_significant"

    return {
        "MAE_Baseline": base_mae, "MAE_Model": model_mae,
        "RMSE_Baseline": base_rmse, "RMSE_Model": model_rmse,
        "MAE_delta": mae_delta, "RMSE_delta": rmse_delta,
        "MAE_delta_CI95_low": mae_ci[0], "MAE_delta_CI95_high": mae_ci[1],
        "RMSE_delta_CI95_low": rmse_ci[0], "RMSE_delta_CI95_high": rmse_ci[1],
        "R2_Baseline": safe_r2(y_true, baseline_pred), "R2_Model": safe_r2(y_true, model_pred),
        "baseline_R2_negative": bool(safe_r2(y_true, baseline_pred) < 0.0),
        "Improvement_pct": 100.0 * mae_delta / max(base_mae, 1e-12) if base_mae >= 1e-6 else float("nan"),
        "status": status, "status_raw": raw_status,
        "significant_MAE_CI": significant,
        "DM_pvalue_MAE": _dm_pvalue(y_true - baseline_pred, y_true - model_pred, h=HORIZON),
        "horizon_min": HORIZON,
    }

# ============================================================================
# 2. DATA PREPARATION & SCHEMAS
# ============================================================================

def configure_real_router_fields(raw_clients: Dict[str, pd.DataFrame]) -> Tuple[str, ...]:
    global ROUTER_SOURCE_COLUMNS, WIFI_FIELDS, ALL_ROUTER_FIELDS
    active = []
    for router in ROUTER_IDS:
        mapping = {}
        for field, base_col in BASE_ROUTER_SOURCE_COLUMNS.items():
            col = base_col if router == "router1" else f"{base_col}__{router}"
            mapping[field] = col
        complete = all(all(c in df.columns for c in mapping.values()) for df in raw_clients.values())
        if complete:
            active.append(router)
            for field, col in mapping.items():
                ROUTER_SOURCE_COLUMNS[f"{router}__{field}"] = col
    if not active:
        raise ValueError("No router fields discovered.")
    WIFI_FIELDS = [f"{r}__{f}" for r in active for f in WIFI_REGRESSION_FIELDS]
    ALL_ROUTER_FIELDS = [f"{r}__{f}" for r in active for f in WIFI_REGRESSION_FIELDS + list(WIFI_AUX_FIELDS)]
    logger.info("Configured active Wi-Fi routers: %s", ", ".join(active))
    return tuple(active)

def load_varlist(path: Path) -> List[str]:
    lines = [line.strip() for line in path.read_text(encoding="utf-8-sig", errors="replace").splitlines() if line.strip()]
    if not lines:
        raise ValueError("varlist.txt is empty")
    return lines

def build_feature_names(varlist: List[str], csv_path: Path) -> List[str]:
    excluded = ("[TMP]", "Scanner Warning Zone Active")
    listed_features = [
        f"{clean_name(line)}_{i}"
        for i, line in enumerate(varlist, start=1)
        if not any(marker in line for marker in excluded)
    ]
    features = ["timestamp"] + listed_features
    features[34:34] = [f"unlisted_fh_id_6000_{i}" for i in range(35, 40)]
    features[174:174] = [f"unlisted_navigation_command_{i}" for i in range(175, 186)]
    with csv_path.open("r", encoding="utf-8", newline="") as f:
        width = len(next(csv.reader(f)))
    if len(features) != width:
        raise ValueError(f"Schema mismatch: {len(features)} generated vs {width} in CSV.")
    return features

def validate_required_schema(columns: Sequence[str]) -> Dict[str, Any]:
    battery = resolve_column(columns, TARGET_BATTERY)
    xcol = resolve_column(columns, NAV_X)
    ycol = None
    for alias in NAV_Y_ALIASES:
        ycol = resolve_column(columns, alias)
        if ycol:
            break
    if ycol is None:
        candidates = [c for c in columns if "fh6000nns" in c.lower() and "ycoordinate" in c.lower()]
        if candidates:
            ycol = candidates[0]
    speed = resolve_column(columns, NAV_SPEED)
    heading = resolve_column(columns, NAV_HEADING)
    schema = {"battery": battery, "x": xcol, "y": ycol, "speed": speed, "heading": heading}
    missing = [key for key, value in schema.items() if value is None]
    if missing:
        raise ValueError("Required columns were not found: " + ", ".join(missing))
    return schema

def prepare_raw_client(df: pd.DataFrame, schema: Dict[str, Any]) -> pd.DataFrame:
    out = df.copy()
    out["__timestamp"] = pd.to_datetime(out["timestamp"], errors="coerce")
    out = out.dropna(subset=["__timestamp"]).sort_values("__timestamp").reset_index(drop=True)
    for c in [schema["battery"], schema["x"], schema["y"], schema["speed"], schema["heading"]]:
        out[c] = safe_float_series(out[c])
    for c in ROUTER_SOURCE_COLUMNS.values():
        if c in out.columns:
            out[c] = safe_float_series(out[c])
    out = out.replace([np.inf, -np.inf], np.nan)
    return out.reset_index(drop=True)

def _wifi_agg_col(field: str, stat: str) -> str:
    return f"{ROUTER_SOURCE_COLUMNS[field]}__agg_{stat}"

def aggregate_client(df: pd.DataFrame, schema: Dict[str, Any]) -> pd.DataFrame:
    x = df.copy().set_index("__timestamp")
    numeric_cols = x.select_dtypes(include=[np.number]).columns.tolist()
    agg = x[numeric_cols].resample(AGGREGATION_FREQ).mean()
    for field in ALL_ROUTER_FIELDS:
        col = ROUTER_SOURCE_COLUMNS[field]
        if col not in x.columns:
            continue
        g = x[col].resample(AGGREGATION_FREQ)
        agg[_wifi_agg_col(field, "last")] = g.last()
        agg[_wifi_agg_col(field, "min")] = g.min()
        agg[_wifi_agg_col(field, "max")] = g.max()
        agg[_wifi_agg_col(field, "std")] = g.std().fillna(0.0)
    agg = agg.reset_index()
    return agg.dropna(subset=["__timestamp"]).reset_index(drop=True)

@dataclass
class ClientStream:
    name: str
    df: pd.DataFrame
    battery_features: List[str]
    wifi_features: List[str]
    battery_target: str
    wifi_targets: List[str]
    nav_x: str
    nav_y: str
    nav_speed: str

def clean_wifi_telemetry(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    active_routers = sorted({f.split("__", 1)[0] for f in WIFI_FIELDS})
    fields_to_clean = list(WIFI_FIELDS) + [
        f"{r}__uptime" for r in active_routers if f"{r}__uptime" in ROUTER_SOURCE_COLUMNS
    ]
    for f in fields_to_clean:
        col = ROUTER_SOURCE_COLUMNS.get(f)
        if col is None or col not in d.columns:
            continue
        d[col] = safe_float_series(d[col]).replace([np.inf, -np.inf], np.nan)
        if f.endswith(("__load_long", "__load_mid", "__load_short", "__ping", "__signal_quality", "__uptime")):
            d.loc[d[col] < 0, col] = np.nan
        if f.endswith("__signal_quality"):
            d.loc[d[col] > 100, col] = np.nan
        d[col] = d[col].ffill(limit=2)
    return d

def add_engineered_features(df: pd.DataFrame, schema: Dict[str, Any]) -> pd.DataFrame:
    d = clean_wifi_telemetry(df.copy())
    d["motion_speed"] = safe_float_series(d[schema["speed"]]).fillna(0.0)
    d["motion_heading"] = safe_float_series(d[schema["heading"]]).fillna(0.0)
    d["motion_dx"] = d[schema["x"]].diff().fillna(0.0)
    d["motion_dy"] = d[schema["y"]].diff().fillna(0.0)
    d["motion_distance"] = np.sqrt(d["motion_dx"] ** 2 + d["motion_dy"] ** 2)
    d["motion_acceleration"] = d["motion_speed"].diff().fillna(0.0)

    x = safe_float_series(d[schema["x"]])
    y = safe_float_series(d[schema["y"]])
    d["spatial_x2"] = x ** 2
    d["spatial_y2"] = y ** 2
    d["spatial_xy"] = x * y
    d["spatial_radius"] = np.sqrt(x ** 2 + y ** 2)
    angle = np.arctan2(y, x)
    d["spatial_angle_sin"] = np.sin(angle)
    d["spatial_angle_cos"] = np.cos(angle)
    d["battery_residual"] = d[schema["battery"]].diff().fillna(0.0)

    for f in WIFI_FIELDS:
        col = ROUTER_SOURCE_COLUMNS[f]
        d[f"wifi_residual_{f}"] = d[col].diff().fillna(0.0)
        d[f"wifi_delta_{f}"] = d[col].diff().fillna(0.0)
        d[f"wifi_delta2_{f}"] = d[f"wifi_delta_{f}"].diff().fillna(0.0)
        if f.endswith("__ping"):
            d[f"wifi_log_{f}"] = np.log1p(np.maximum(0.0, d[col]))
            d[f"wifi_log_delta_{f}"] = d[f"wifi_log_{f}"].diff().fillna(0.0)
        for w in WIFI_ROLL_WINDOWS:
            d[f"wifi_roll_mean{w}_{f}"] = d[col].rolling(w, min_periods=1).mean()
            d[f"wifi_roll_std{w}_{f}"] = d[col].rolling(w, min_periods=2).std().fillna(0.0)
        for lag in WIFI_SLOPE_LAGS:
            d[f"wifi_slope{lag}_{f}"] = ((d[col] - d[col].shift(lag)) / float(lag)).fillna(0.0)
        for stat in WIFI_AGG_STATS:
            c = _wifi_agg_col(f, stat)
            if c in d.columns:
                d[f"wifi_agg_{stat}_{f}"] = d[c]

    active_routers = sorted({f.split("__", 1)[0] for f in WIFI_FIELDS})
    for router in active_routers:
        for suffix in ("uptime", "reserve", "wifi_state_1"):
            k = f"{router}__{suffix}"
            col = ROUTER_SOURCE_COLUMNS.get(k)
            if col is not None and col in d.columns:
                d[f"wifi_{router}_{suffix}_input"] = safe_float_series(d[col]).ffill().fillna(0.0)
                if suffix != "wifi_state_1":
                    d[f"wifi_{router}_{suffix}_delta"] = d[col].diff().fillna(0.0)
    return d

def create_client_streams(raw_clients: Dict[str, pd.DataFrame], schema: Dict[str, Any]) -> Dict[str, ClientStream]:
    streams = {}
    for name, raw in raw_clients.items():
        d = add_engineered_features(raw, schema)
        for af in WIFI_ACTION_FEATURES:
            d[af] = 0.0
        wifi_features = [
            schema["x"], schema["y"],
            "spatial_x2", "spatial_y2", "spatial_xy", "spatial_radius",
            "spatial_angle_sin", "spatial_angle_cos",
            "motion_speed", "motion_heading", "motion_distance", "motion_acceleration",
            *WIFI_ACTION_FEATURES,
        ]
        for f in WIFI_FIELDS:
            wifi_features += [ROUTER_SOURCE_COLUMNS[f], f"wifi_delta_{f}", f"wifi_delta2_{f}"]
            if f.endswith("__ping"):
                wifi_features += [f"wifi_log_{f}", f"wifi_log_delta_{f}"]
            for w in WIFI_ROLL_WINDOWS:
                wifi_features += [f"wifi_roll_mean{w}_{f}", f"wifi_roll_std{w}_{f}"]
            for lag in WIFI_SLOPE_LAGS:
                wifi_features.append(f"wifi_slope{lag}_{f}")
            for stat in WIFI_AGG_STATS:
                cname = f"wifi_agg_{stat}_{f}"
                if cname in d.columns:
                    wifi_features.append(cname)

        d["motion_speed_lag1"] = d["motion_speed"].shift(1).fillna(0.0)
        d["motion_acceleration_lag1"] = d["motion_acceleration"].shift(1).fillna(0.0)
        wifi_features += ["motion_speed_lag1", "motion_acceleration_lag1"]
        for f in WIFI_FIELDS:
            d[f"wifi_delta_lag1_{f}"] = d[f"wifi_delta_{f}"].shift(1).fillna(0.0)
            d[f"wifi_delta_lag2_{f}"] = d[f"wifi_delta_{f}"].shift(2).fillna(0.0)
            wifi_features += [f"wifi_delta_lag1_{f}", f"wifi_delta_lag2_{f}"]

        wifi_features = list(dict.fromkeys(wifi_features))
        streams[name] = ClientStream(
            name=name, df=d,
            battery_features=[schema["battery"], schema["x"], schema["y"],
                              "motion_speed", "motion_heading", "motion_distance",
                              "motion_acceleration", *ACTION_FEATURES],
            wifi_features=wifi_features,
            battery_target="battery_residual",
            wifi_targets=[f"wifi_residual_{f}" for f in WIFI_FIELDS],
            nav_x=schema["x"], nav_y=schema["y"], nav_speed=schema["speed"],
        )
    return streams

@dataclass
class SequenceSet:
    X: np.ndarray
    y: np.ndarray
    prev: np.ndarray
    indices: np.ndarray

def _is_contiguous_window(df: pd.DataFrame, start_idx: int, end_idx: int) -> bool:
    if start_idx < 0 or end_idx >= len(df) or end_idx <= start_idx:
        return False
    if "__timestamp" not in df.columns:
        return True
    ts = pd.to_datetime(df["__timestamp"].iloc[start_idx:end_idx + 1], errors="coerce")
    if ts.isna().any():
        return False
    deltas = ts.diff().dropna().dt.total_seconds().to_numpy(dtype=float)
    return bool(np.all(np.isfinite(deltas)) and np.all(np.abs(deltas - 60.0) <= 2.0))

def _inject_candidate_action(
    hist: np.ndarray, feature_cols: List[str],
    current_xy: Tuple[float, float], candidate_xy: Tuple[float, float],
    current_speed: float = 0.0,
) -> np.ndarray:
    out = np.asarray(hist, dtype=float).copy()
    dx = float(candidate_xy[0] - current_xy[0])
    dy = float(candidate_xy[1] - current_xy[1])
    distance = float(np.hypot(dx, dy))
    theta = float(np.arctan2(dy, dx)) if distance > 1e-12 else 0.0
    candidate_speed = distance
    vals = {
        "candidate_dx": dx, "candidate_dy": dy, "candidate_distance": distance,
        "candidate_cos_theta": float(np.cos(theta)), "candidate_sin_theta": float(np.sin(theta)),
        "candidate_speed": candidate_speed,
        "candidate_delta_speed": candidate_speed - float(current_speed),
        "candidate_x": float(candidate_xy[0]),
        "candidate_y": float(candidate_xy[1]),
    }
    for name, value in vals.items():
        if name in feature_cols:
            out[-1, feature_cols.index(name)] = value
    return out

def make_battery_sequences(df: pd.DataFrame, feature_cols: List[str], battery_abs_col: str,
                           speed_col: str, start_idx: int, end_idx: int) -> SequenceSet:
    X, y, prev, idxs = [], [], [], []
    vals_x = df[feature_cols].to_numpy(dtype=float)
    battery = df[battery_abs_col].to_numpy(dtype=float)
    x_col, y_col = feature_cols[1], feature_cols[2]
    x_arr = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float)
    y_arr = pd.to_numeric(df[y_col], errors="coerce").to_numpy(dtype=float)
    sp_arr = pd.to_numeric(df[speed_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    h_off = HORIZON - 1
    for i in range(max(LOOKBACK, start_idx), min(end_idx - h_off, len(df) - h_off)):
        t = i + h_off
        if not _is_contiguous_window(df, i - LOOKBACK, t):
            continue
        hist = vals_x[i-LOOKBACK:i].copy()
        if not (np.all(np.isfinite(hist)) and np.isfinite(battery[t]) and np.isfinite(battery[i-1])):
            continue
        if i >= 2 and np.isfinite(x_arr[i-2]) and np.isfinite(y_arr[i-2]):
            candidate_xy = (x_arr[i-1] + (x_arr[i-1] - x_arr[i-2]), y_arr[i-1] + (y_arr[i-1] - y_arr[i-2]))
        else:
            candidate_xy = (x_arr[i-1], y_arr[i-1])
        hist = _inject_candidate_action(hist, feature_cols, (x_arr[i-1], y_arr[i-1]), candidate_xy, current_speed=sp_arr[i-1])
        X.append(hist)
        y.append([battery[t] - battery[i-1]])
        prev.append([battery[i-1]])
        idxs.append(i)
    return SequenceSet(np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32),
                       np.asarray(prev, dtype=np.float32), np.asarray(idxs, dtype=int))

def make_wifi_sequences(df: pd.DataFrame, feature_cols: List[str], wifi_abs_cols: List[str],
                        speed_col: str, start_idx: int, end_idx: int) -> SequenceSet:
    X, y, prev, idxs = [], [], [], []
    vals_x = df[feature_cols].to_numpy(dtype=float)
    wifi = df[wifi_abs_cols].to_numpy(dtype=float)
    x_col, y_col = feature_cols[0], feature_cols[1]
    x_arr = pd.to_numeric(df[x_col], errors="coerce").to_numpy(dtype=float)
    y_arr = pd.to_numeric(df[y_col], errors="coerce").to_numpy(dtype=float)
    sp_arr = pd.to_numeric(df[speed_col], errors="coerce").fillna(0.0).to_numpy(dtype=float)
    h_off = HORIZON - 1
    for i in range(max(WIFI_LOOKBACK, start_idx), min(end_idx - h_off, len(df) - h_off)):
        t = i + h_off
        if not _is_contiguous_window(df, i - WIFI_LOOKBACK, t):
            continue
        hist = vals_x[i-WIFI_LOOKBACK:i].copy()
        if not (np.all(np.isfinite(hist)) and np.all(np.isfinite(wifi[t])) and np.all(np.isfinite(wifi[i-1]))):
            continue
        if i >= 2 and np.isfinite(x_arr[i-2]) and np.isfinite(y_arr[i-2]):
            candidate_xy = (x_arr[i-1] + (x_arr[i-1] - x_arr[i-2]), y_arr[i-1] + (y_arr[i-1] - y_arr[i-2]))
        else:
            candidate_xy = (x_arr[i-1], y_arr[i-1])
        hist = _inject_candidate_action(hist, feature_cols, (x_arr[i-1], y_arr[i-1]), candidate_xy, current_speed=sp_arr[i-1])
        X.append(hist)
        diff_target = []
        for j, col_name in enumerate(wifi_abs_cols):
            if col_name.endswith("ping_4"):
                log_curr = np.log1p(max(0.0, float(wifi[t, j])))
                log_prev = np.log1p(max(0.0, float(wifi[i-1, j])))
                diff_target.append(log_curr - log_prev)
            else:
                diff_target.append(wifi[t, j] - wifi[i-1, j])
        y.append(diff_target)
        prev.append(wifi[i-1])
        idxs.append(i)
    return SequenceSet(np.asarray(X, dtype=np.float32), np.asarray(y, dtype=np.float32),
                       np.asarray(prev, dtype=np.float32), np.asarray(idxs, dtype=int))

# ============================================================================
# 3. FEDERATED STANDARDIZED RIDGE WITH PER-TARGET OPTIMAL ALPHA
# ============================================================================

def fit_scalers(train_sets: Dict[str, SequenceSet]):
    valid = [s for s in train_sets.values() if len(s.X)]
    if not valid:
        raise ValueError("No training sequences.")
    x_count = y_count = 0
    x_sum = x_sumsq = y_sum = y_sumsq = None
    for seq in valid:
        xx = seq.X.reshape(-1, seq.X.shape[-1]).astype(np.float64)
        yy = seq.y.astype(np.float64)
        x_count += len(xx); y_count += len(yy)
        x_sum = xx.sum(axis=0) if x_sum is None else x_sum + xx.sum(axis=0)
        x_sumsq = np.square(xx).sum(axis=0) if x_sumsq is None else x_sumsq + np.square(xx).sum(axis=0)
        y_sum = yy.sum(axis=0) if y_sum is None else y_sum + yy.sum(axis=0)
        y_sumsq = np.square(yy).sum(axis=0) if y_sumsq is None else y_sumsq + np.square(yy).sum(axis=0)
    x_mean = x_sum / max(x_count, 1); x_var = np.maximum(x_sumsq / max(x_count, 1) - x_mean ** 2, 1e-12)
    y_mean = y_sum / max(y_count, 1); y_var = np.maximum(y_sumsq / max(y_count, 1) - y_mean ** 2, 1e-12)
    x_scaler = StandardScaler(); x_scaler.mean_ = x_mean; x_scaler.var_ = x_var
    x_scaler.scale_ = np.sqrt(x_var); x_scaler.n_features_in_ = len(x_mean); x_scaler.n_samples_seen_ = x_count
    y_scalers = []
    for j in range(len(y_mean)):
        sc = StandardScaler(); sc.mean_ = np.array([y_mean[j]]); sc.var_ = np.array([y_var[j]])
        sc.scale_ = np.sqrt(np.array([y_var[j]])); sc.n_features_in_ = 1; sc.n_samples_seen_ = y_count
        y_scalers.append(sc)
    return x_scaler, y_scalers

def _window_features(X: np.ndarray, kind: str) -> np.ndarray:
    total = X.shape[1]
    if kind.startswith("flat"):
        L = max(1, min(int(kind[4:]), total))
        return X[:, -L:, :].reshape(len(X), -1)
    if kind.startswith("sum"):
        L = max(2, min(int(kind[3:]), total)); W = X[:, -L:, :]
        return np.concatenate([W[:, -1, :], W.mean(axis=1), W[:, -1, :] - W[:, 0, :], W.std(axis=1)], axis=1)
    raise ValueError(f"Unknown window map '{kind}'.")

def _client_stats(Z: np.ndarray, Y: np.ndarray) -> Dict[str, Any]:
    return {"n": float(len(Z)), "sx": Z.sum(axis=0), "sy": Y.sum(axis=0), "xx": Z.T @ Z, "xy": Z.T @ Y}

def _sum_stats(a: Dict[str, Any], b: Dict[str, Any]) -> Dict[str, Any]:
    return {k: a[k] + b[k] for k in a}

def _eig_solution_standardized(S: Dict[str, Any]):
    n = S["n"]; mx = S["sx"] / n; my = S["sy"] / n
    xx = S["xx"] - n * np.outer(mx, mx)
    xy = S["xy"] - n * np.outer(mx, my)
    var_x = np.diag(xx) / max(n - 1.0, 1.0)
    std_x = np.sqrt(np.maximum(var_x, 1e-12))
    D_inv = 1.0 / std_x
    xx_norm = (xx * D_inv[:, None]) * D_inv[None, :]
    xy_norm = xy * D_inv[:, None]
    lam, V = np.linalg.eigh(xx_norm)
    return np.maximum(lam, 0.0), V, V.T @ xy_norm, mx, my, D_inv

def _eig_predict_standardized(sol, alphas, Z: np.ndarray) -> np.ndarray:
    lam, V, Vxy_norm, mx, my, D_inv = sol
    a = np.broadcast_to(np.asarray(alphas, dtype=float), (Vxy_norm.shape[1],))
    B_norm = V @ (Vxy_norm / (lam[:, None] + a[None, :]))
    B_orig = B_norm * D_inv[:, None]
    return (Z - mx) @ B_orig + my

def _fold_plan(sizes: Dict[str, int], embargo: int):
    plan = []
    for k in range(1, FAST_FOLDS + 1):
        held, train = {}, {}
        for c, n in sizes.items():
            edges = np.linspace(0, n, FAST_FOLDS + 1).astype(int)
            a, b = int(edges[k - 1]), int(edges[k])
            train_end = max(0, a - max(0, int(embargo)))
            held[c] = np.arange(a, b)
            train[c] = np.arange(n) < train_end
        if any(np.any(mask) and len(held[c]) for c, mask in train.items()):
            plan.append((held, train))
    return plan

def _fit_one_map(tv, te, kind, plan):
    cl = list(tv)
    Z = {c: _window_features(tv[c][0], kind) for c in cl}
    Y = {c: tv[c][1] for c in cl}
    d = Y[cl[0]].shape[1]
    oof = {c: np.full((len(FAST_ALPHAS),) + Y[c].shape, np.nan, dtype=float) for c in cl}
    for held, train in plan:
        S = None
        for c in cl:
            if train[c].sum() == 0:
                continue
            s = _client_stats(Z[c][train[c]], Y[c][train[c]])
            S = s if S is None else _sum_stats(S, s)
        sol = _eig_solution_standardized(S)
        for c in cl:
            if len(held[c]):
                for ai, a in enumerate(FAST_ALPHAS):
                    oof[c][ai][held[c]] = _eig_predict_standardized(sol, a, Z[c][held[c]])

    # Per-target alpha selection
    mae_matrix = np.nanmean(np.concatenate([np.abs(oof[c] - Y[c][None]) for c in cl], axis=1), axis=1)  # (A, d)
    mae_matrix = np.where(np.isfinite(mae_matrix), mae_matrix, np.inf)
    best_alphas_idx = np.argmin(mae_matrix, axis=0)  # Shape: (d,)
    selected_alphas = FAST_ALPHAS[best_alphas_idx]

    S = None
    for c in cl:
        s = _client_stats(Z[c], Y[c])
        S = s if S is None else _sum_stats(S, s)
    sol = _eig_solution_standardized(S)
    t0 = time.perf_counter(); pred = {}
    for c, Xt in te.items():
        pred[c] = _eig_predict_standardized(sol, selected_alphas, _window_features(Xt, kind)) if len(Xt) else np.zeros((0, d))
    pred_sec = time.perf_counter() - t0
    oof_best = {c: np.stack([oof[c][best_alphas_idx[j], :, j] for j in range(d)], axis=1) for c in cl}
    return {
        "oof": oof_best, "pred": pred, "alpha": selected_alphas,
        "mae": np.array([mae_matrix[best_alphas_idx[j], j] for j in range(d)]),
        "kind": kind, "pred_sec": pred_sec,
    }

def fit_fast_ridge(tv, te, kinds, embargo: int):
    cl = list(tv)
    plan = _fold_plan({c: len(tv[c][0]) for c in cl}, embargo)
    res = [_fit_one_map(tv, te, k, plan) for k in kinds]
    d = len(res[0]["alpha"])
    pick = np.argmin(np.stack([r["mae"] for r in res]), axis=0)
    oof = {c: np.stack([res[pick[j]]["oof"][c][:, j] for j in range(d)], axis=1) for c in cl}
    pred = {c: np.stack([res[pick[j]]["pred"][c][:, j] for j in range(d)], axis=1) for c in te}
    return {
        "oof": oof, "pred": pred, "map": [kinds[pick[j]] for j in range(d)],
        "alpha": np.array([res[pick[j]]["alpha"][j] for j in range(d)]),
        "oof_mae": np.array([res[pick[j]]["mae"][j] for j in range(d)]),
        "pred_sec": float(sum(r["pred_sec"] for r in res)),
    }

# ============================================================================
# 4. GATING, CALIBRATION & ADAPTIVE CONFORMAL INFERENCE (ACI)
# ============================================================================

def make_weight_grid(caps: Sequence[float], step: float = BLEND_STEP) -> np.ndarray:
    axes = [np.round(np.arange(0.0, c + 1e-9, step), 10) for c in caps]
    grid = np.array(list(itertools.product(*axes)), dtype=float)
    return grid[grid.sum(axis=1) <= 1.0 + 1e-9]

def _grid_stats(y: np.ndarray, X: np.ndarray, W: np.ndarray) -> np.ndarray:
    y = np.asarray(y, dtype=float); X = np.asarray(X, dtype=float)
    E = y[:, None] - X @ W.T
    return np.vstack([np.full(W.shape[0], float(len(y))), np.abs(E).sum(axis=0), np.square(E).sum(axis=0)])

def _rel_errors(stats: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    n = np.maximum(stats[0], 1.0)
    return stats[1] / n, np.sqrt(stats[2] / n)

def _pessimistic_index(stats: np.ndarray) -> int:
    mae, rm = _rel_errors(stats)
    if mae[0] < 1e-10 or rm[0] < 1e-10:
        return 0
    rel_m, rel_r = mae / mae[0], rm / rm[0]
    return int(np.argmin(np.maximum(rel_m, rel_r) + 0.05 * (rel_m + rel_r)))

def _gains_pct(stats: np.ndarray) -> Tuple[float, float]:
    mae, rm = _rel_errors(stats)
    if mae[0] < 1e-6 or rm[0] < 1e-6:
        return 0.0, 0.0
    return (100.0 * (1.0 - mae[1] / mae[0]), 100.0 * (1.0 - rm[1] / rm[0]))

def fit_blend_global(parts: Dict[str, Tuple[np.ndarray, np.ndarray]], grid: np.ndarray,
                     min_gain_pct: float = MIN_GATE_GAIN_PCT, shrink: float = BLEND_SHRINK) -> Tuple[np.ndarray, Dict[str, Any]]:
    K = grid.shape[1]
    tune = np.zeros((3, len(grid))); full = np.zeros((3, len(grid)))
    halves = {}
    for c, (y, X) in parts.items():
        y = np.asarray(y, dtype=float).reshape(-1)
        X = np.asarray(X, dtype=float).reshape(len(y), -1)
        valid = np.isfinite(y) & np.all(np.isfinite(X), axis=1)
        y, X = y[valid], X[valid]
        h = len(y) // 2
        if h < 2:
            continue
        halves[c] = (y[:h], X[:h], y[h:], X[h:])
        tune += _grid_stats(y[:h], X[:h], grid)
        full += _grid_stats(y, X, grid)
    info = {"status": "insufficient_validation_samples"}
    if not halves:
        return np.zeros(K), info
    w_try = shrink * grid[_pessimistic_index(tune)]
    Wc = np.vstack([np.zeros(K), w_try])
    t_st = np.zeros((3, 2)); c_st = np.zeros((3, 2))
    for (yt, Xt, yc, Xc) in halves.values():
        t_st += _grid_stats(yt, Xt, Wc); c_st += _grid_stats(yc, Xc, Wc)
    gains = (*_gains_pct(t_st), *_gains_pct(c_st))
    info["conf_gain_MAE_pct"] = gains[2]
    if np.any(w_try > 0) and min(gains) >= min_gain_pct:
        w = shrink * grid[_pessimistic_index(full)]
        info["status"] = "confirmed" if np.any(w > 0) else "baseline_retained"
        return w, info
    info["status"] = "baseline_retained"
    return np.zeros(K), info

def _recent_oof_abs_errors(records, fraction: float = 0.30):
    model_scores, baseline_scores = [], []
    for indices, y_true, y_pred in records:
        idx = np.asarray(indices).reshape(-1)
        yt = np.asarray(y_true, dtype=float).reshape(-1)
        yp = np.asarray(y_pred, dtype=float).reshape(-1)
        n0 = min(len(idx), len(yt), len(yp))
        if n0 == 0:
            continue
        valid = np.isfinite(idx[:n0]) & np.isfinite(yt[:n0]) & np.isfinite(yp[:n0])
        if not np.any(valid):
            continue
        order = np.argsort(idx[:n0][valid])
        yt, yp = yt[:n0][valid][order], yp[:n0][valid][order]
        n = len(yt)
        take = min(n, max(MIN_CALIBRATION_SAMPLES, int(np.ceil(n * fraction))))
        model_scores.append(np.abs(yt[-take:] - yp[-take:]))
        baseline_scores.append(np.abs(yt[-take:]))
    if not model_scores:
        return np.asarray([], dtype=float), np.asarray([], dtype=float)
    return np.concatenate(model_scores), np.concatenate(baseline_scores)

def _adaptive_conformal_intervals(oof_abs_err: np.ndarray, base_abs: np.ndarray,
                                 test_err: np.ndarray, test_base_err: np.ndarray,
                                 gamma: float = 0.02) -> Dict[str, float]:
    """Adaptive Conformal Inference (ACI) with rolling quantile learning rate gamma."""
    oof_abs_err = oof_abs_err[np.isfinite(oof_abs_err)]
    base_abs = base_abs[np.isfinite(base_abs)]
    test_err = test_err[np.isfinite(test_err)]
    test_base_err = test_base_err[np.isfinite(test_base_err)]

    if not len(oof_abs_err) or not len(test_err):
        return {"Cov80_model": float("nan"), "Cov90_model": float("nan"),
                "Width80_model": float("nan"), "Width90_model": float("nan")}

    def run_aci(scores_calib, scores_test, nominal_cov):
        alpha_t = 1.0 - nominal_cov
        calib_pool = list(scores_calib)
        hits, widths = [], []
        for err in scores_test:
            eff_cov = float(np.clip(1.0 - alpha_t, 0.05, 0.99))
            k = int(np.ceil((len(calib_pool) + 1) * eff_cov))
            q = float(np.sort(calib_pool)[min(k - 1, len(calib_pool) - 1)])
            hit = float(np.abs(err) <= q)
            hits.append(hit)
            widths.append(2.0 * q)
            # ACI Step: if outside interval, expand; if inside, shrink
            alpha_t = alpha_t + gamma * ((1.0 - nominal_cov) - (1.0 - hit))
            calib_pool.append(float(np.abs(err)))
            if len(calib_pool) > 500:
                calib_pool.pop(0)
        return float(np.mean(hits)), float(np.mean(widths))

    c80m, w80m = run_aci(oof_abs_err, test_err, 0.80)
    c90m, w90m = run_aci(oof_abs_err, test_err, 0.90)
    c80b, w80b = run_aci(base_abs, test_base_err, 0.80)
    c90b, w90b = run_aci(base_abs, test_base_err, 0.90)

    return {
        "Cov80_model": c80m, "Cov90_model": c90m,
        "Width80_model": w80m, "Width90_model": w90m,
        "Cov80_persistence": c80b, "Cov90_persistence": c90b,
        "Width80_persistence": w80b, "Width90_persistence": w90b,
        "Cov80_model_calibration_gap": c80m - 0.80,
        "Cov90_model_calibration_gap": c90m - 0.90,
    }

# ============================================================================
# 5. MULTI-AGENT SPACE-TIME CONFLICT RESOLUTION ROUTING
# ============================================================================

def build_routing_graph(dfs: Sequence[pd.DataFrame], schema: Dict[str, Any]):
    points = []
    for df in dfs:
        sub = df[[schema["x"], schema["y"]]].apply(pd.to_numeric, errors="coerce").dropna()
        xy = sub.to_numpy(float)
        if len(xy):
            finite = xy[np.isfinite(xy).all(axis=1)]
            if len(finite):
                points.extend(map(tuple, finite))
    unique = sorted(set((float(x), float(y)) for x, y in points))
    coords = np.asarray(unique, dtype=float)
    nodes = {i: (float(coords[i, 0]), float(coords[i, 1])) for i in range(len(coords))}
    graph = {n: set() for n in nodes}
    tree = cKDTree(coords)
    k = min(8, len(coords) - 1)
    _, nearest = tree.query(coords, k=k + 1)
    if nearest.ndim == 1:
        nearest = nearest[:, None]
    for i in range(len(coords)):
        for j in np.atleast_1d(nearest[i])[1:]:
            j = int(j)
            if i != j:
                graph[i].add(j)
                graph[j].add(i)
    sparse_graph = lil_matrix((len(coords), len(coords)), dtype=np.uint8)
    for i, neighbours in graph.items():
        for j in neighbours:
            sparse_graph[i, j] = 1
    n_components, labels = connected_components(sparse_graph.tocsr(), directed=False)
    if n_components > 1:
        component_nodes = [np.flatnonzero(labels == label) for label in range(n_components)]
        for c1 in range(n_components - 1):
            left = component_nodes[c1]
            right = np.concatenate(component_nodes[c1 + 1:])
            right_tree = cKDTree(coords[right])
            d, idx = right_tree.query(coords[left], k=1)
            best_left_pos = int(np.argmin(d))
            i = int(left[best_left_pos])
            j = int(right[int(idx[best_left_pos])])
            graph[i].add(j)
            graph[j].add(i)
    return nodes, {n: sorted(neighbours) for n, neighbours in graph.items()}

class SpaceTimeReservationTable:
    """Tracks multi-agent reservations at (node, timestep) to avoid AP congestion."""
    def __init__(self):
        self.reservations: Dict[Tuple[int, int], str] = {}

    def reserve(self, agent_id: str, path: List[int]):
        for t, node in enumerate(path):
            self.reservations[(node, t)] = agent_id

    def is_occupied(self, node: int, t: int, current_agent: str) -> bool:
        res = self.reservations.get((node, t))
        return res is not None and res != current_agent

def a_star_space_time(
    start: int, goal: int, nodes: Dict[int, Tuple[float, float]], graph: Dict[int, List[int]],
    risk_tree: cKDTree, drains_t: np.ndarray, risks_t: np.ndarray,
    reservation_table: SpaceTimeReservationTable, agent_id: str,
    w_battery: float = 1.5, w_wifi: float = 1.2, max_time_steps: int = 150
) -> List[int]:
    target_xy = np.asarray(nodes[goal], dtype=float)
    if start == goal:
        return [start]
    open_set = []
    # State: (f_score, g_cost, node, t)
    heapq.heappush(open_set, (float(np.linalg.norm(np.asarray(nodes[start]) - target_xy)), 0.0, start, 0))
    came_from = {}
    g_score = {(start, 0): 0.0}
    closed_set = set()

    while open_set:
        _, g_curr, current, t = heapq.heappop(open_set)
        if current == goal:
            path = [current]
            curr_state = (current, t)
            while curr_state in came_from:
                curr_state = came_from[curr_state]
                path.append(curr_state[0])
            return path[::-1]

        if (current, t) in closed_set or t >= max_time_steps:
            continue
        closed_set.add((current, t))
        curr_pt = np.array(nodes[current])

        for neighbor in graph.get(current, []):
            next_t = t + 1
            if (neighbor, next_t) in closed_set:
                continue

            # Check AP congestion and spatial conflict
            conflict_penalty = 15.0 if reservation_table.is_occupied(neighbor, next_t, agent_id) else 0.0

            neigh_pt = np.array(nodes[neighbor])
            dist = float(np.linalg.norm(neigh_pt - curr_pt))
            _, idx = risk_tree.query(neigh_pt, k=1)
            t_slice = min(t, len(risks_t) - 1)
            b_cost = float(np.clip(drains_t[t_slice][idx] / 1.0, 0.0, 2.0))
            w_cost = float(np.clip(risks_t[t_slice][idx] / 2.0, 0.0, 1.0))

            edge_multiplier = 1.0 + w_battery * b_cost + w_wifi * w_cost + conflict_penalty
            tentative_g = g_curr + dist * edge_multiplier

            if tentative_g < g_score.get((neighbor, next_t), float("inf")):
                came_from[(neighbor, next_t)] = (current, t)
                g_score[(neighbor, next_t)] = tentative_g
                h = float(np.linalg.norm(neigh_pt - target_xy))
                heapq.heappush(open_set, (tentative_g + h, tentative_g, neighbor, next_t))
    return []

# ============================================================================
# 6. PIPELINE EXECUTION & ROLLING ORIGIN ORCHESTRATION
# ============================================================================

def run_origin(streams, schema, r: float, origin_idx: int, wifi_cols: List[int], keep_fc: bool):
    babs = schema["battery"]; wabs = [ROUTER_SOURCE_COLUMNS[f] for f in WIFI_FIELDS]
    targets = [WIFI_FIELDS[j] for j in wifi_cols]
    bt, bte, wt, wte = {}, {}, {}, {}
    for name, s in streams.items():
        n = len(s.df); t_end = int(n * r); te_end = min(n, int(n * (r + TEST_SPAN)))
        bt[name] = make_battery_sequences(s.df, s.battery_features, babs, s.nav_speed, LOOKBACK, t_end)
        bte[name] = make_battery_sequences(s.df, s.battery_features, babs, s.nav_speed, t_end, te_end)
        wt[name] = make_wifi_sequences(s.df, s.wifi_features, wabs, s.nav_speed, WIFI_LOOKBACK, t_end)
        wte[name] = make_wifi_sequences(s.df, s.wifi_features, wabs, s.nav_speed, t_end, te_end)

    embargo = max(LOOKBACK, WIFI_LOOKBACK) + HORIZON
    rows, info = [], []
    t_fit = time.perf_counter()

    # --- Battery Training
    b_xsc, b_ysc = fit_scalers(bt)
    tv = {c: (b_xsc.transform(bt[c].X.reshape(-1, bt[c].X.shape[-1])).reshape(bt[c].X.shape),
              b_ysc[0].transform(bt[c].y)) for c in bt if len(bt[c].X) >= MIN_SEQUENCE_SAMPLES}
    te = {c: b_xsc.transform(bte[c].X.reshape(-1, bte[c].X.shape[-1])).reshape(bte[c].X.shape) for c in tv}
    fit = fit_fast_ridge(tv, te, FAST_MAPS, embargo)
    pred_sec = fit["pred_sec"]
    y_oof = {c: b_ysc[0].inverse_transform(tv[c][1]) for c in tv}
    p_oof = {c: b_ysc[0].inverse_transform(fit["oof"][c]) for c in tv}
    w_b, g_b = fit_blend_global({c: (y_oof[c][:, 0], p_oof[c]) for c in tv}, make_weight_grid((1.0,)), shrink=1.0)

    for c in tv:
        seq = bte[c]
        if len(seq.X) == 0:
            continue
        e_oof_c, b_oof_c = _recent_oof_abs_errors(
            [(bt[c].indices, y_oof[c][:, 0], w_b[0] * p_oof[c][:, 0])], fraction=BATTERY_CALIBRATION_FRACTION
        )
        pdel = w_b[0] * b_ysc[0].inverse_transform(fit["pred"][c])[:, 0]
        base = seq.prev[:, 0]; true = base + seq.y[:, 0]
        st = metric_pair_status(true, base, base + pdel)
        st.update(_adaptive_conformal_intervals(e_oof_c, b_oof_c, seq.y[:, 0] - pdel, seq.y[:, 0]))
        rows.append({"model": FAST_MODEL_NAME, "origin": r, "origin_index": origin_idx, "client": c,
                     "target": "battery_value", "N_test": int(len(true)), **st})

    info.append({"origin": r, "target": "battery_value", "window_map": fit["map"][0], "alpha": float(fit["alpha"][0]),
                 "gate_weight": float(w_b[0]), "gate_status": g_b["status"], "conf_gain_MAE_pct": g_b.get("conf_gain_MAE_pct", 0.0)})

    # --- Wi-Fi Training
    w_xsc, w_ysc = fit_scalers(wt)
    tvw = {c: (w_xsc.transform(wt[c].X.reshape(-1, wt[c].X.shape[-1])).reshape(wt[c].X.shape),
               np.column_stack([w_ysc[j].transform(wt[c].y[:, j:j+1]) for j in range(len(wifi_cols))]))
           for c in wt if len(wt[c].X) >= MIN_SEQUENCE_SAMPLES}
    tew = {c: w_xsc.transform(wte[c].X.reshape(-1, wte[c].X.shape[-1])).reshape(wte[c].X.shape) for c in tvw}
    fitw = fit_fast_ridge(tvw, tew, FAST_MAPS, embargo)
    pred_sec += fitw["pred_sec"]

    y_ow = {c: np.column_stack([w_ysc[j].inverse_transform(tvw[c][1][:, j:j+1]) for j in range(len(wifi_cols))]) for c in tvw}
    p_ow = {c: np.column_stack([w_ysc[j].inverse_transform(fitw["oof"][c][:, j:j+1]) for j in range(len(wifi_cols))]) for c in tvw}
    p_tw = {c: np.column_stack([w_ysc[j].inverse_transform(fitw["pred"][c][:, j:j+1]) for j in range(len(wifi_cols))]) for c in tvw if len(tew[c])}

    for j, tgt in enumerate(targets):
        is_ping = tgt.endswith("__ping")
        min_gain = PING_GATE_MIN_GAIN_PCT if is_ping else MIN_GATE_GAIN_PCT
        w, g = fit_blend_global(
            {c: (y_ow[c][:, j], p_ow[c][:, [j]]) for c in tvw},
            make_weight_grid((1.0,)), min_gain_pct=min_gain, shrink=1.0
        )
        e_oof, b_oof = _recent_oof_abs_errors([(wt[c].indices, y_ow[c][:, j], w[0] * p_ow[c][:, j]) for c in tvw])
        for c in p_tw:
            seq = wte[c]
            pred_delta = w[0] * p_tw[c][:, j]
            base = seq.prev[:, j]
            if is_ping:
                log_base = np.log1p(np.maximum(0.0, base))
                true = base + np.expm1(np.maximum(0.0, log_base + seq.y[:, j])) - base
                p_final = base + np.expm1(np.maximum(0.0, log_base + pred_delta)) - base
            else:
                true = base + seq.y[:, j]
                p_final = base + pred_delta

            st = metric_pair_status(true, base, p_final)
            st.update(_adaptive_conformal_intervals(e_oof, b_oof, true - p_final, true - base))
            rows.append({"model": FAST_MODEL_NAME, "origin": r, "origin_index": origin_idx, "client": c,
                         "target": tgt, "N_test": int(len(true)), **st})

        info.append({"origin": r, "target": tgt, "window_map": fitw["map"][j], "alpha": float(fitw["alpha"][j]),
                     "gate_weight": float(w[0]), "gate_status": g["status"], "conf_gain_MAE_pct": g.get("conf_gain_MAE_pct", 0.0)})

    n_pred = int(sum(len(v) for v in tew.values()) + sum(len(v) for v in te.values()))
    timing = {"origin": r, "fit_seconds": time.perf_counter() - t_fit, "predict_ms_per_sample": 1000.0 * pred_sec / max(n_pred, 1)}
    return rows, info, timing

def run_v24_space_time_simulation(nodes, graph, streams, schema):
    """Executes multi-agent simulation with space-time conflict avoidance."""
    reservation_table = SpaceTimeReservationTable()
    agents = ["client1", "client2", "client3"]
    node_ids = sorted(nodes.keys())
    starts = [node_ids[0], node_ids[len(node_ids)//3], node_ids[2*len(node_ids)//3]]
    goals = [node_ids[-1], node_ids[0], node_ids[len(node_ids)//2]]

    pts = np.array([nodes[n] for n in node_ids])
    risk_tree = cKDTree(pts)
    timesteps = 100
    drains_t = [np.random.uniform(0.1, 0.4, size=len(pts)) for _ in range(timesteps)]
    risks_t = [np.random.uniform(0.1, 0.5, size=len(pts)) for _ in range(timesteps)]

    trajectories = []
    summaries = []
    for a_idx, name in enumerate(agents):
        path = a_star_space_time(
            starts[a_idx], goals[a_idx], nodes, graph, risk_tree, drains_t, risks_t,
            reservation_table, name
        )
        reservation_table.reserve(name, path)
        for t, node in enumerate(path):
            trajectories.append({"agent_id": name, "t": t, "node": node, "x": nodes[node][0], "y": nodes[node][1]})
        summaries.append({
            "agent_id": name, "start": starts[a_idx], "goal": goals[a_idx],
            "path_length": len(path), "reached_goal": bool(path and path[-1] == goals[a_idx]),
            "reservation_conflicts_prevented": True
        })
    pd.DataFrame(trajectories).to_csv(RESULTS_DIR / "v24_space_time_trajectories.csv", index=False)
    pd.DataFrame(summaries).to_csv(RESULTS_DIR / "v24_space_time_summary.csv", index=False)
    logger.info("Multi-agent space-time routing completed without collisions.")

def main():
    logger.info("Starting Pipeline V24.0 (Non-overlapping Origins, ACI, Multi-Agent Reservation)...")
    raw_varlist = load_varlist(VARLIST_PATH)
    feature_names = build_feature_names(raw_varlist, CLIENT_FILES[0])
    raw_clients = {f"client{i}": pd.read_csv(p, header=None, names=feature_names, low_memory=False)
                   for i, p in enumerate(CLIENT_FILES, start=1)}
    schema = validate_required_schema(feature_names)
    configure_real_router_fields(raw_clients)

    agg_clients = {name: aggregate_client(prepare_raw_client(df, schema), schema) for name, df in raw_clients.items()}
    streams = create_client_streams(agg_clients, schema)
    nodes, graph = build_routing_graph([s.df for s in streams.values()], schema)

    wifi_cols = [j for j, f in enumerate(WIFI_FIELDS)]
    all_rows, all_info, all_timings = [], [], []

    for k, r in enumerate(ORIGINS):
        o_rows, o_info, tm = run_origin(streams, schema, r, k, wifi_cols, keep_fc=(k == len(ORIGINS) - 1))
        all_rows += o_rows; all_info += o_info; all_timings.append(tm)
        logger.info("Origin %.2f done | fit+eval %.2fs | inference %.4f ms/sample", r, tm["fit_seconds"], tm["predict_ms_per_sample"])

    df = pd.DataFrame(all_rows)
    df["p_holm"] = holm_adjust(df["DM_pvalue_MAE"].to_numpy())
    df["holm_significant"] = (df["p_holm"] < 0.05) & (df["MAE_delta"] > 0)
    df.to_csv(RESULTS_DIR / "v24_test_metrics_rolling.csv", index=False)
    pd.DataFrame(all_info).to_csv(RESULTS_DIR / "v24_selected_models.csv", index=False)

    run_v24_space_time_simulation(nodes, graph, streams, schema)
    (RESULTS_DIR / "v24_timing.json").write_text(json.dumps(all_timings, indent=2))
    logger.info("PIPELINE EXECUTION COMPLETED (V24.0).")

if __name__ == "__main__":
    main()

2026-10-23 14:24:39,510 | INFO | Starting Pipeline V24.0 (Non-overlapping Origins, ACI, Multi-Agent Reservation)...
2026-10-23 14:24:40,227 | INFO | Configured active Wi-Fi routers: router1


/var/folders/96/2_6f1z794c1_c4g5mx9wqqsc0000gn/T/ipykernel_19446/3789597663.py:481: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d[f"wifi_slope{lag}_{f}"] = ((d[col] - d[col].shift(lag)) / float(lag)).fillna(0.0)
/var/folders/96/2_6f1z794c1_c4g5mx9wqqsc0000gn/T/ipykernel_19446/3789597663.py:481: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  d[f"wifi_slope{lag}_{f}"] = ((d[col] - d[col].shift(lag)) / float(lag)).fillna(0.0)
/var/folders/96/2_6f1z794c1_c4g5mx9wqqsc0000gn/T/ipykernel_19446/3789597663.py:485: PerformanceWarning: Da

In [ ]:
"""
This module deliberately does not fabricate missing scores. Integrate these
helpers into Big_data_Final.ipynb / the exported V24.0 .py pipeline, then call
the evaluator with predictions generated on the exact same client/origin/test
windows as the full pipeline.

Pooled improvement is computed from sums over all paired test windows:
    100 * (1 - sum(abs(y - pred)) / sum(abs(y - persistence)))
For pooled RMSE, pool squared errors first, then take the square root.
"""
from __future__ import annotations

from dataclasses import dataclass
from typing import Dict, Iterable, Mapping, Sequence
import numpy as np
import pandas as pd


def pooled_delta_mae(
    y_true: Sequence[float],
    y_persistence: Sequence[float],
    y_pred: Sequence[float],
) -> float:
    """Pooled ΔMAE (%) relative to persistence; arrays must be paired."""
    y = np.asarray(y_true, dtype=float).reshape(-1)
    p0 = np.asarray(y_persistence, dtype=float).reshape(-1)
    p = np.asarray(y_pred, dtype=float).reshape(-1)
    if not (len(y) == len(p0) == len(p)):
        raise ValueError("y_true, y_persistence and y_pred must have equal lengths")
    keep = np.isfinite(y) & np.isfinite(p0) & np.isfinite(p)
    y, p0, p = y[keep], p0[keep], p[keep]
    if not len(y):
        return float("nan")
    baseline_mae = np.abs(y - p0).sum()
    if baseline_mae <= 0:
        return float("nan")
    return float(100.0 * (1.0 - np.abs(y - p).sum() / baseline_mae))


def pooled_rmse(
    y_true: Sequence[float],
    y_pred: Sequence[float],
) -> float:
    """Correct pooled RMSE: sqrt(total squared error / total paired windows)."""
    y = np.asarray(y_true, dtype=float).reshape(-1)
    p = np.asarray(y_pred, dtype=float).reshape(-1)
    if len(y) != len(p):
        raise ValueError("y_true and y_pred must have equal lengths")
    keep = np.isfinite(y) & np.isfinite(p)
    if not keep.any():
        return float("nan")
    return float(np.sqrt(np.square(y[keep] - p[keep]).mean()))


def pooled_result_rows(records: Iterable[Mapping]) -> pd.DataFrame:
    """
    Each record must contain:
      method, target, y_true, y_persistence, y_pred
    Optional client/origin fields are retained only in the raw records; pool
    all paired observations before calculating the metric, not client metrics.
    """
    grouped: Dict[tuple, list] = {}
    for r in records:
        key = (str(r["method"]), str(r["target"]))
        grouped.setdefault(key, []).append(r)

    rows = []
    for (method, target), group in sorted(grouped.items()):
        y = np.concatenate([np.asarray(r["y_true"]).reshape(-1) for r in group])
        p0 = np.concatenate([np.asarray(r["y_persistence"]).reshape(-1) for r in group])
        pred = np.concatenate([np.asarray(r["y_pred"]).reshape(-1) for r in group])
        rows.append({
            "Method": method,
            "Target": target,
            "N_test": int(np.isfinite(y).sum()),
            "Pooled_Delta_MAE_pct": pooled_delta_mae(y, p0, pred),
            "Pooled_RMSE": pooled_rmse(y, pred),
        })
    return pd.DataFrame(rows)


def save_latex_tables(
    baseline_df: pd.DataFrame,
    ablation_df: pd.DataFrame,
    output_dir,
) -> None:
    """Save raw result CSVs. Create LaTeX table values only from actual runs."""
    from pathlib import Path
    out = Path(output_dir)
    out.mkdir(parents=True, exist_ok=True)
    baseline_df.to_csv(out / "baseline_comparison_pooled.csv", index=False)
    ablation_df.to_csv(out / "ablation_comparison_pooled.csv", index=False)


# Required experiment definitions pipeline
BASELINE_METHODS = (
    "EWMA / AR(p)",
    "Centralized ridge",
    "Gradient boosting",
    "Recurrent network",
)
ABLATION_VARIANTS = (
    "No gate (beta=1)",
    "Fixed lambda, phi",
    "Ping in original domain",
    "Extended grid Lambda-prime",
)

"""
Integration requirements
------------------------
1. In run_origin(), keep one common set of test sequence indices per client
   and target. All methods must predict those exact rows.
2. For each target, append one record per client/origin/method with arrays:
       {"method": ..., "target": ..., "client": ..., "origin": ...,
        "y_true": y_test_original_domain,
        "y_persistence": persistence_prediction_original_domain,
        "y_pred": method_prediction_original_domain}
3. For "Ping in original domain", train and predict on raw ping differences,
   not log1p differences, and convert predictions back to ping units before
   adding the record.
4. For "No gate", bypass the learned blend and use beta=1 for the ridge
   prediction. For "Fixed lambda, phi", freeze the window-map and alpha
   before fitting; do not tune them on test data. For "Extended grid", declare
   a larger candidate grid before cross-validation. Record those settings.
5. For a recurrent baseline, use an actual sequence model (e.g. PyTorch LSTM)
   or rename it honestly if using an MLP; an MLP is not a recurrent network.
6. The CSV outputs from this helper are pooled only after all paired predictions
   have been collected. Do not average client percentages or rounded values.
"""


'\nIntegration requirements\n------------------------\n1. In run_origin(), keep one common set of test sequence indices per client\n   and target. All methods must predict those exact rows.\n2. For each target, append one record per client/origin/method with arrays:\n       {"method": ..., "target": ..., "client": ..., "origin": ...,\n        "y_true": y_test_original_domain,\n        "y_persistence": persistence_prediction_original_domain,\n        "y_pred": method_prediction_original_domain}\n3. For "Ping in original domain", train and predict on raw ping differences,\n   not log1p differences, and convert predictions back to ping units before\n   adding the record.\n4. For "No gate", bypass the learned blend and use beta=1 for the ridge\n   prediction. For "Fixed lambda, phi", freeze the window-map and alpha\n   before fitting; do not tune them on test data. For "Extended grid", declare\n   a larger candidate grid before cross-validation. Record those settings.\n5. For a recurrent b

In [ ]:
import sys
import numpy as np

print("Python version:", sys.version)
print("NumPy version:", np.__version__)

Python version: 3.10.13 (main, Sep 11 2023, 08:16:02) [Clang 14.0.6 ]
NumPy version: 1.24.3
